# CSC 240 Final Project: Data Preprocessing
arXiv Papers Before and After the Release of ChatGPT


In [ ]:
import json
from email.utils import parsedate_to_datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "arxiv-metadata-oai-snapshot.json"   

PRE_START, PRE_END   = pd.Timestamp("2020-01-01"), pd.Timestamp("2022-12-01")  # Jan 2020 – Nov 2022
POST_START, POST_END = pd.Timestamp("2023-07-01"), pd.Timestamp("2025-10-01")  # Jul 2023 – Sep 2025

# ChatGPT's public release date
CHATGPT = pd.Timestamp("2022-11-30")

# Dataset overview
The file has one paper per line. Because it is too large to load at once, we read it line by line and keep only what we need for counting: each paper's ID, main field, the date it was first posted (v1), and the date of its latest version

In [ ]:
# arXiv has dozens of physics categories (hep-th, cond-mat, astro-ph, ...)
# we group them all as "physics" so we can compare broad fields
PHYSICS = ("astro-ph", "cond-mat", "gr-qc", "hep-", "math-ph", "nlin", "nucl-", "physics", "quant-ph")

def field_of(categories):

    # a paper can list several categories, e.g. "cs.LG stat.ML"; the first one is its main one
    primary = categories.split()[0]
    archive = primary.split(".")[0] # keep only the part before the dot: "cs.LG" -> "cs"
    if archive.startswith(PHYSICS):
        return "physics"
    return archive  # otherwise: cs, math, stat, eess, econ, q-bio, or q-fin

def to_date(s):
    return parsedate_to_datetime(s).replace(tzinfo=None)

meta = []        # 1 row of basic info per paper
columns = None   # dataset's column names, taken from the 1st paper
n_rows = 0       # numbers of papers read so far


with open(DATA_PATH) as f:
    for line in f:
        p = json.loads(line)
        n_rows += 1
        if columns is None:
            columns = list(p.keys())
        
        # "versions" is the paper's edit history:
        # 1st entry is when it was first posted (v1), the last is its latest update
        v1 = to_date(p["versions"][0]["created"])
        vlast = to_date(p["versions"][-1]["created"])
        fld = field_of(p["categories"])

        # save: ID, field, first-posted date, last-updated date, # of versions
        meta.append((p["id"], fld, v1, vlast, len(p["versions"])))
    
# summary
meta = pd.DataFrame(meta, columns=["id", "field", "v1", "vlast", "n_versions"])
print(f"Rows (papers): {n_rows:,}")
print(f"Columns ({len(columns)}): {columns}")
print(f"v1 date range: {meta.v1.min().date()} to {meta.v1.max().date()}")

# Papers by field and period (dated by v1)

In [ ]:
# label each paper by when it was first posted:
# "pre" = before-ChatGPT window, "post" = after-ChatGPT window, None = outside both
def period_of(d):
    if PRE_START <= d < PRE_END: return "pre"
    if POST_START <= d < POST_END: return "post"
    return None

meta["period"] = meta.v1.map(period_of)
win = meta.dropna(subset=["period"])

#count papers per field and period
counts = (win.groupby(["field", "period"]).size().unstack()[["pre", "post"]]
          .sort_values("post", ascending=False))

# the windows have different lengths (35 months before, 27 after),
# so compare papers PER MONTH: how much faster are papers coming in after ChatGPT?
PRE_MONTHS, POST_MONTHS = 35, 27
counts["growth_%"] = (100 * ((counts.post / POST_MONTHS) / (counts.pre / PRE_MONTHS) - 1)).round(1)

# total row across all fields (also per month)
total_pre, total_post = counts.pre.sum(), counts.post.sum()
counts.loc["TOTAL"] = [total_pre, total_post,
                       round(100 * ((total_post / POST_MONTHS) / (total_pre / PRE_MONTHS) - 1), 1)]

# show counts as whole numbers 
counts.astype({"pre": int, "post": int})

In [ ]:
# count new papers per month (by first-posted date), starting from 2018
monthly = meta[meta.v1 >= "2018-01-01"].set_index("v1").resample("MS").size()

fig, ax = plt.subplots(figsize=(7, 3)) 
monthly.plot(ax=ax) # the line: papers per month

# red dashed line on the day ChatGPT was released
ax.axvline(CHATGPT, color="red", ls="--", label="ChatGPT release")

# shade the two time windows
ax.axvspan(PRE_START, PRE_END, color="tab:blue", alpha=0.12, label="Before window")
ax.axvspan(POST_START, POST_END, color="tab:orange", alpha=0.12, label="After window")

ax.set_title("Monthly New arXiv Submissions, 2018–2026")
ax.set_ylabel("New papers per month"); ax.set_xlabel("")
ax.legend()
plt.savefig("fig_monthly_submissions.png", dpi=200)

## version check
The Kaggle snapshot stores the **latest** abstract. A pre-period paper revised after Nov 2022 may carry a post-ChatGPT abstract, so we flag and exclude those from the pre group

In [ ]:
# all papers first posted in the "before ChatGPT" window
pre = win[win.period == "pre"]

# True/False for each paper: was its latest version posted after ChatGPT's release?
# If yes, the abstract we have might have been rewritten with ChatGPT's help
revised_after = (pre.vlast >= CHATGPT)

# Overall numbers
# (Averaging True/False values gives the fraction that are True, e.g. 0.115 -> 11.5%)
print(f"Pre-period papers: {len(pre):,}")
print(f"  revised after ChatGPT release: {revised_after.sum():,} ({100*revised_after.mean():.1f}%)")

# same % for each field, smallest to largest
(pre.assign(revised_after=revised_after)
    .groupby("field").revised_after.mean()
    .mul(100).round(1).sort_values()
    .rename("% revised after ChatGPT"))

# Data preprocessing
1. Drop pre-window papers revised after ChatGPT (version check above)
2. Pull the abstracts of the remaining papers
3. Clean the text (remove LaTeX markup, standardize)
4. Drop abstracts we can't use
5. Save for feature engineering

In [ ]:
# keep papers in the two windows, minus pre papers revised after ChatGPT
keep = win[~((win.period == "pre") & (win.vlast >= CHATGPT))]
keep_ids = set(keep.id)
print(f"Papers kept: {len(keep):,}")

## Pull abstracts
Read the file again and keep the title, abstract, and comments of the kept papers

In [ ]:
rows = []
with open(DATA_PATH) as f:
    for line in f:
        p = json.loads(line)
        if p["id"] in keep_ids:
            rows.append((p["id"], p["title"], p["abstract"], p["comments"] or ""))

text = pd.DataFrame(rows, columns=["id", "title", "abstract_raw", "comments"])
df = keep[["id", "field", "period", "v1", "vlast", "n_versions"]].merge(text, on="id")
print(f"Abstracts pulled: {len(df):,}")

## Clean text
- LaTeX math (`$...$`) → `[MATH]`; the math itself is kept in `math`, since numbers like `$95\%$` count toward specificity
- citations (`\cite{}`) removed, URLs → `[URL]`
- accents and escapes fixed (`\"o` → o, `\%` → %), other LaTeX commands and braces removed
- line breaks and extra spaces removed

In [ ]:
import re, unicodedata

MATH = re.compile(r"\$\$.+?\$\$|\$.+?\$|\\\(.+?\\\)|\\\[.+?\\\]", re.S)
CITE = re.compile(r"~?\\(?:cite[tp]?|ref|eqref|label)\*?\{[^}]*\}")
URL = re.compile(r"https?://\S+|www\.\S+|\\url\{[^}]*\}")
ACCENT = re.compile(r"\\[\"'`^~=.Hcuvk]\s*\{?\\?([A-Za-z])\}?")   # \"o -> o
COMMAND = re.compile(r"\\[A-Za-z]+\*?")                            # \emph, \textbf, ...

def clean(s):
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s)                  # abstracts are hard-wrapped: join the lines
    math = " ||| ".join(m.strip() for m in MATH.findall(s))
    s = MATH.sub(" [MATH] ", s)
    s = CITE.sub(" ", s)
    s = URL.sub(" [URL] ", s)
    s = s.replace(r"\%", "%").replace(r"\&", "&").replace(r"\_", "_")
    s = ACCENT.sub(r"\1", s)
    s = COMMAND.sub(" ", s)
    s = re.sub(r"[{}]", "", s).replace("~", " ")
    s = re.sub(r"-{2,3}", "-", s)
    s = re.sub(r"\s+([.,;:)])", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s, math

cleaned = df.abstract_raw.map(clean)
df["abstract"] = cleaned.str[0]
df["math"] = cleaned.str[1]

# word count, not counting the [MATH]/[URL] placeholders
WORD = re.compile(r"[A-Za-z]+(?:['\-][A-Za-z]+)*")
df["n_words"] = df.abstract.str.replace(r"\[(?:MATH|URL)\]", " ", regex=True).map(lambda s: len(WORD.findall(s)))

# one example, before and after
ex = df[df.math != ""].iloc[0]
print("RAW:  ", " ".join(ex.abstract_raw.split())[:300])
print("CLEAN:", ex.abstract[:300])

## Drop abstracts we can't use

In [ ]:
# share of words that are common English function words (low for non-English abstracts)
FUNCTION_WORDS = set("the of and a an in to is we for that with on by this are as be from our which it at these can".split())
def english_share(s):
    w = WORD.findall(s.lower())
    return np.mean([x in FUNCTION_WORDS for x in w]) if w else 0

rules = {
    "withdrawn":   df.comments.str.contains("withdrawn", case=False)
                   | ((df.n_words < 40) & df.abstract_raw.str.contains(r"withdr[ae]w", case=False)),
    "too short (<30 words)": df.n_words < 30,
    "non-English": df.abstract.map(english_share) < 0.15,
    "duplicate":   df.abstract.str.lower().duplicated(),
}

# apply the rules in order and count how many each one removes
steps = [("pulled", len(df))]
for name, bad in rules.items():
    bad = bad.reindex(df.index)
    steps.append((name, -int(bad.sum())))
    df = df[~bad]
steps.append(("final", len(df)))
pd.DataFrame(steps, columns=["step", "papers"])

## Variables
| Variable | Description |
|---|---|
| `id` | arXiv ID |
| `field` | main field from the primary category (cs, physics, math, eess, stat, q-bio, econ, q-fin) |
| `period` | `pre` (Jan 2020–Nov 2022) or `post` (Jul 2023–Sep 2025), by v1 date |
| `v1`, `vlast`, `n_versions` | first-posted date, latest-version date, number of versions |
| `title` | paper title |
| `abstract` | cleaned abstract, **use this for features** |
| `math` | LaTeX math removed from the abstract |
| `n_words` | words in the cleaned abstract |

In [ ]:
# final papers by field and period
final = df.groupby(["field", "period"]).size().unstack()[["pre", "post"]].sort_values("post", ascending=False)
final.loc["TOTAL"] = final.sum()
print(final)

cols = ["id", "field", "period", "v1", "vlast", "n_versions", "title", "abstract", "math", "n_words"]
df[cols].to_csv("abstracts_clean.csv.gz", index=False)

## Introductions (H2)
The Kaggle file has no full text, so introductions come from arXiv directly:
1. pick 60 papers per field and period, same version as the stored abstract
2. download each paper's LaTeX source (PDF if the authors didn't upload source)
3. cut the text from `\section{Introduction}` to the next `\section` (PDF: from "Introduction" to the next numbered heading)
4. clean it with the same `clean()` as the abstracts

Needs internet and `pip install requests pypdf`. ~3 s per paper (arXiv's rate limit), so 960 papers ≈ 50 min; downloads are saved in `intro_cache/`, so re-runs are fast.

In [ ]:
intro = df.sample(frac=1, random_state=240).groupby(["field", "period"]).head(60)
print(f"Introduction subsample: {len(intro):,} papers")

In [ ]:
import os, io, re, gzip, tarfile, time
import requests
from pypdf import PdfReader

os.makedirs("intro_cache", exist_ok=True)

def download(paper_id, version):
    """Get the paper's source from arXiv (LaTeX, or PDF if no source). Saved to intro_cache/ so re-runs don't re-download."""
    path = f"intro_cache/{paper_id.replace('/', '_')}v{version}"
    if os.path.exists(path):
        return open(path, "rb").read()
    r = requests.get(f"https://export.arxiv.org/e-print/{paper_id}v{version}",
                     headers={"User-Agent": "CSC240 class project, University of Rochester"}, timeout=60)
    time.sleep(3)                       # arXiv asks for ~3 seconds between requests
    if r.status_code != 200:
        return None
    open(path, "wb").write(r.content)
    return r.content

def tex_files(data):
    """Unpack the source: a .tar.gz of files, or a single gzipped .tex file."""
    try:
        with tarfile.open(fileobj=io.BytesIO(data), mode="r:*") as tar:
            return {m.name: tar.extractfile(m).read().decode("utf-8", "ignore")
                    for m in tar.getmembers() if m.isfile() and m.name.endswith(".tex")}
    except tarfile.TarError:
        try:
            return {"main.tex": gzip.decompress(data).decode("utf-8", "ignore")}
        except OSError:
            return {"main.tex": data.decode("utf-8", "ignore")}

def full_tex(files):
    """Main file = the one with \\begin{document}; paste in \\input{}/\\include{} files."""
    strip = lambda s: re.sub(r"(?<!\\)%.*", "", s)          # remove LaTeX comments
    files = {k: strip(v) for k, v in files.items()}
    main = next((v for v in files.values() if r"\begin{document}" in v), None)
    if main is None:
        return ""
    by_name = {re.sub(r"\.tex$", "", os.path.normpath(k)): v for k, v in files.items()}
    for _ in range(3):                                       # nested \input up to 3 levels
        main = re.sub(r"\\(?:input|include)\{([^}]+)\}",
                      lambda m: by_name.get(re.sub(r"\.tex$", "", os.path.normpath(m.group(1))), ""), main)
    return main

SECTION = re.compile(r"\\section\*?\s*(?:\[[^\]]*\])?\s*\{([^}]*)\}")
END = re.compile(r"\\(?:section|appendix|bibliography|begin\{thebibliography\}|end\{document\}|acknowledg)", re.I)

def intro_from_tex(tex):
    """Text between \\section{Introduction} and the next \\section.
    Papers without sections (e.g. PRL letters): text between the abstract and the first section."""
    for m in SECTION.finditer(tex):
        if "introduction" in m.group(1).lower():
            end = END.search(tex, m.end())
            return tex[m.end(): end.start() if end else None], "latex"
    start = max(tex.find(r"\end{abstract}"), tex.find(r"\maketitle"))
    if start > 0:
        end = END.search(tex, start)
        return tex[start: end.start() if end else None].replace(r"\end{abstract}", "").replace(r"\maketitle", ""), "latex (no heading)"
    return "", None

def drop_environments(t):
    """Remove figures/tables; turn display math into $...$ so clean() handles it."""
    t = re.sub(r"\\begin\{(figure|table|tabular|wrapfigure|algorithm)\*?\}.*?\\end\{\1\*?\}", " ", t, flags=re.S)
    t = re.sub(r"\\begin\{(equation|align|eqnarray|gather|multline)\*?\}(.*?)\\end\{\1\*?\}", r" $\2$ ", t, flags=re.S)
    t = re.sub(r"\\(?:sub)+section\*?\{[^}]*\}|\\paragraph\{[^}]*\}", " ", t)
    t = re.sub(r"\\footnote\{[^{}]*\}", " ", t)
    t = re.sub(r"\\(?:label|item)\b(\{[^}]*\})?", " ", t)
    t = re.sub(r"\\(?:begin|end)\{[^}]*\}(?:\[[^\]]*\])?", " ", t)    # leftover \begin{itemize} etc.
    return t

PDF_START = re.compile(r"(?im)^\s*(?:1\.?|I\.)?\s*introduction\s*$")
PDF_END = re.compile(r"(?m)^\s*(?:2\.?|II\.)\s+[A-Z][A-Za-z ,:&-]{2,60}$")

def intro_from_pdf(data):
    """Fallback: read the first 4 pages, cut from the 'Introduction' heading to the next numbered heading."""
    text = "\n".join(p.extract_text() or "" for p in PdfReader(io.BytesIO(data)).pages[:4])
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)             # re-join hyphenated line breaks
    m = PDF_START.search(text)
    if not m:
        return "", None
    end = PDF_END.search(text, m.end())
    return text[m.end(): end.start() if end else m.end() + 8000], "pdf"

def get_intro(paper_id, version):
    data = download(paper_id, version)
    if data is None:
        return "", "download failed"
    if data[:4] == b"%PDF":
        return intro_from_pdf(data)
    body, how = intro_from_tex(full_tex(tex_files(data)))
    return drop_environments(body), how


In [ ]:
# try a few papers first: set N_TEST = None to run all of them
N_TEST = 10

rows = []
todo = intro if N_TEST is None else intro.head(N_TEST)
for i, r in enumerate(todo.itertuples(), 1):
    try:
        body, how = get_intro(r.id, r.n_versions)
    except Exception as e:
        body, how = "", f"error: {type(e).__name__}"
    text = clean(body)[0] if body else ""
    rows.append((r.id, how, text))
    if i % 50 == 0:
        print(f"{i}/{len(todo)}")

intros = pd.DataFrame(rows, columns=["id", "intro_source", "introduction"])
intros["intro_words"] = intros.introduction.map(lambda s: len(WORD.findall(s)))
intros = intro[["id", "field", "period", "n_versions", "abstract"]].merge(intros, on="id")

# how many worked, by source
print(intros.intro_source.value_counts(dropna=False))
intros[["id", "field", "period", "intro_source", "intro_words", "introduction"]].head()

In [ ]:
# keep usable introductions (at least 100 words) and save
ok = intros[intros.intro_words >= 100]
print(ok.groupby(["field", "period"]).size().unstack()[["pre", "post"]])
ok.to_csv("introductions.csv", index=False)